<a href="https://colab.research.google.com/github/camille2019/Women-Capital-Trial-Analysis/blob/main/clean_and_stem_transcripts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspellchecker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 53.4 MB/s eta 0:00:00


In [2]:
!pip install wordfreq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 7.4 MB/s eta 0:00:00


In [102]:
import pandas as pd
import io
import numpy as np
import sys
import re
import os
import re

import seaborn as sns
import matplotlib.pyplot as plt

from scipy import stats

import nltk
nltk.download('punkt')
from nltk.tokenize import word_tokenize
import pickle
import json

from pathlib import Path
import spacy

from multiprocessing import Pool
from multiprocessing import cpu_count
import math
import time
import random
from nltk.stem import PorterStemmer


from collections import Counter
from pathlib import Path
import re

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm
from nltk.corpus import wordnet


import json
import os
import glob

import spacy
import en_core_web_sm

import snowballstemmer



[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [129]:
from nltk.corpus import words
nltk.download('words')


[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Package words is already up-to-date!


True

In [104]:

  >>> import nltk
  >>> nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [146]:
from google.colab import drive
drive.mount('/content/drive/')


Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [106]:
mens_transcripts_full_path = Path('/content/drive/MyDrive/Capital_trials/men')
womens_transcripts_full_path = Path('/content/drive/MyDrive/Capital_trials/women')
corrected_mens_transcripts_path = Path('/content/drive/MyDrive/Capital_trials/men/corrected_docs')
corrected_womens_transcripts_path = Path('/content/drive/MyDrive/Capital_trials/women/corrected_docs')

Create word dictionary



In [107]:
nlp = spacy.load("en_core_web_sm")
nlp = en_core_web_sm.load()

In [108]:
#check legal terms with lexpredict legal dictionary https://github.com/LexPredict/lexpredict-legal-dictionary/blob/master/en/legal/common_US_terms_top1000.csv
legal_terms_df = pd.read_csv('/content/drive/MyDrive/Capital_trials/common_US_terms_top1000.csv')
legal_list = legal_terms_df.Term.to_list()

legal_terms2 = pd.read_csv('/content/drive/MyDrive/Capital_trials/common_law.csv')
legal_list2 = legal_terms_df.Term.to_list()


In [109]:
#drug terms vocab created using https://nida.nih.gov/research-topics/drugs-a-to-z

drug_terms = pd.read_csv('/content/drive/MyDrive/Capital_trials/drug_names_set.csv', header=None)

drug_list = drug_terms.values.tolist()

In [110]:
state_names = pd.read_csv('/content/drive/MyDrive/Capital_trials/states.csv')
states_list =state_names.State.tolist()

In [111]:
#markdown coutnry list from https://developers.google.com/public-data/docs/canonical/countries_csv
with open('/content/drive/MyDrive/Capital_trials/countries.md', 'r') as f:
    lines = f.readlines()
    content = "".join([line for line in lines if not set(line.strip()).issubset({'|', '-', ' '})])
df= pd.read_csv(io.StringIO(content), sep="|").dropna(axis=1, how='all')
df.columns = df.columns.str.strip()
countries = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)


/tmp/ipykernel_9137/2294585849.py:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  countries = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)


In [126]:
def_names = pd.read_csv("/content/Data Collection-Nathalie's view (1).csv")
def_names_list =def_names.Name.tolist()
def_names_list = [word for string in def_names_list for word in string.split()]
def_names_list = [re.sub(r'[^\w\s]', '', s) for s in def_names_list]


In [112]:
countries_list = countries.name.tolist()

In [113]:
drug_list = [s.strip() for s in drug_list[0]]

In [130]:
nltk_vocab = set(w.lower() for w in words.words())
legal_vocab = (set(l.lower() for l in legal_list) - nltk_vocab)
legal_vocab2 = set(w.lower() for w in legal_list2)
wordnet_vocab = set(w.lower() for w in nltk.corpus.words.words())
spacy_vocab = {lex.text.lower() for lex in nlp.vocab if lex.is_alpha}
drug_vocab = set(d.lower() for d in drug_list)

countries = set(country.lower() for country in countries_list)
states = set(state.lower() for state in states_list)

names = set(name.lower() for name in def_names_list)

In [131]:
vocab = nltk_vocab | legal_vocab |wordnet_vocab | spacy_vocab | legal_vocab2 | drug_vocab | countries | states | names

Additional Cleaning Steps (after running OCR correction)

In [28]:
#apply basic cleaning to each transcript in a
def full_basic_clean(transcripts_path, output_folder_json, output_folder_stem):

  for filename in  os.listdir(transcripts_path):
    full_path= transcripts_path + '/' +filename

    name , _ , txt = filename.partition('.')

    new_file_path = output_folder_json + '/' +name +'.json'
    lower_tokenize_regex(full_path, new_file_path)
    #stem the transcript

    stem_json(new_file_path,output_folder_stem)


In [60]:
def lower_tokenize_regex(transcript_path, clean_transcript_pathname):

  with open(transcript_path, 'r', encoding='utf8', errors='replace') as infile:
    text_data = infile.read()
    text_data = text_data.lower()
    #cleaned_text = re.sub(r"[^a-z ]", '', text_data)
    # cleaned_text = re.sub(r"[\p{L}'\s’]+" , '', text_data) #include apostraphes and accent marks
    cleaned_text =  re.sub(r"[^A-Za-zÁÉÍÓÚáéíóúÑñÜü\s’]+", " ", text_data)
    cleaned_text = word_tokenize(cleaned_text)

  with open(clean_transcript_pathname, "w", encoding="utf-8") as outfile:
    json.dump(cleaned_text, outfile)

In [61]:
def stem_json(clean_transcript_pathname, output_path):
  stemmer = PorterStemmer()

  with open(clean_transcript_pathname, 'r', encoding='utf8', errors='replace') as infile:

    transcript = json.load(infile)
    stemmed_words = [stemmer.stem(word) for word in transcript]

  with open(output_path, "w", encoding="utf-8") as outfile:
    json.dump(stemmed_words, outfile)

In [143]:
#assign 2 transcripts as variables for test evaluation
women_test1_pre ='/content/drive/MyDrive/Capital_trials/women/Christa Pike.txt'
women_test2_pre ='/content/drive/MyDrive/Capital_trials/women/Socorro Caro.txt'

men_test1_pre = '/content/drive/MyDrive/Capital_trials/men/Christopher Sepulvado Combined Transcript.txt'
men_test2_pre = '/content/drive/MyDrive/Capital_trials/men/Fabian Hernandez Combined Transcript.txt'


women_test1_post = '/content/drive/MyDrive/Capital_trials/women/corrected_docs/Christa Pike.txt'
women_test2_post = '/content/drive/MyDrive/Capital_trials/women/corrected_docs/Socorro Caro.txt'

men_test1_post = '/content/drive/MyDrive/Capital_trials/men/corrected_docs/Christopher Sepulvado Combined Transcript.txt'
men_test2_post ='/content/drive/MyDrive/Capital_trials/men/corrected_docs/Fabian Hernandez Combined Transcript.txt'

pre_ocr = '/content/drive/MyDrive/Capital_trials/test/preocr/'
post_ocr = '/content/drive/MyDrive/Capital_trials/test/postocr/'


In [63]:
token_output = '/content/drive/MyDrive/Capital_trials/test/postocr/Christa Pike.json'
lower_tokenize_regex(women_test1_post, token_output)
stem_output = '/content/drive/MyDrive/Capital_trials/test/postocr/stem/Christa Pike.json'
stem_json(token_output, stem_output)

In [64]:
token_output = '/content/drive/MyDrive/Capital_trials/test/postocr/Socorro Caro.json'
lower_tokenize_regex(women_test2_post, token_output)
stem_output = '/content/drive/MyDrive/Capital_trials/test/postocr/stem/Socorro Caro.json'
stem_json(token_output, stem_output)

In [65]:
token_output = '/content/drive/MyDrive/Capital_trials/test/postocr/Christopher Sepulvado Combined Transcript.json'
lower_tokenize_regex(men_test1_post, token_output)
stem_output = '/content/drive/MyDrive/Capital_trials/test/postocr/stem/Christopher Sepulvado Combined Transcripto.json'
stem_json(token_output, stem_output)


token_output = '/content/drive/MyDrive/Capital_trials/test/postocr/Fabian Hernandez Combined Transcript.json'
lower_tokenize_regex(men_test2_post, token_output)
stem_output = '/content/drive/MyDrive/Capital_trials/test/postocr/stem/Fabian Hernandez Combined Transcript.json'
stem_json(token_output, stem_output)


In [160]:
!ls "/content/drive/MyDrive/Capital_trials/test/preocr"


'Christopher Sepulvado Combined Transcript.json'
'Fabian Hernandez Combined Transcript.json'


In [161]:
#compare results on preocr
token_output = '/content/drive/MyDrive/Capital_trials/test/preocr/Christopher Sepulvado Combined Transcript.json'
lower_tokenize_regex(men_test1_pre, token_output)

token_output = '/content/drive/MyDrive/Capital_trials/test/preocr/Fabian Hernandez Combined Transcript.json'
lower_tokenize_regex(men_test2_pre, token_output)


In [162]:
peocr_token_output = '/content/drive/MyDrive/Capital_trials/test/preocr/Christa Pike.json'
lower_tokenize_regex('/content/drive/MyDrive/Capital_trials/women/Christa Pike.txt', peocr_token_output)
preocr_ttoken_output = '/content/drive/MyDrive/Capital_trials/test/preocr/Socorro Caro.json'
lower_tokenize_regex('/content/drive/MyDrive/Capital_trials/women/Socorro Caro.txt', preocr_ttoken_output)

In [163]:
folder =  Path('/content/drive/MyDrive/Capital_trials/test/preocr/')
oov_dict = dict()


for file in folder.iterdir():
  with open(file, "r", encoding="utf-8") as f:
    w = json.load(f)
    total_len = len(w)
    json_words = set(w)

  print('file name')
  print(file.name)

  oov_words = json_words - (stem_vocab | vocab | stem_vocab2)
  print('total length')
  print(total_len)

  print('num unique words')
  print(len(json_words))

  print('num unique out of vocab words')
  print(len(oov_words))
  oov_dict[file.name] = oov_words

file name
Christopher Sepulvado Combined Transcript.json
total length
146886
num unique words
5430
num unique out of vocab words
1423
file name
Fabian Hernandez Combined Transcript.json
total length
382769
num unique words
8494
num unique out of vocab words
2698
file name
Christa Pike.json
total length
151994
num unique words
5164
num unique out of vocab words
1544
file name
Socorro Caro.json
total length
2073246
num unique words
15366
num unique out of vocab words
5372


In [132]:
#stem our vocab then compare on the stemmed json
stemmer = PorterStemmer()
stem_vocab = {stemmer.stem(word) for word in vocab}

stemmer2 = snowballstemmer.stemmer('english');
stem_vocab2 = set(stemmer2.stemWord(w) for w in vocab)

In [146]:
folder =  Path('/content/drive/MyDrive/Capital_trials/test/postocr/without_stem')
oov_dict_without_stem = dict()

for file in folder.iterdir():
  with open(file, "r", encoding="utf-8") as f:
    w = json.load(f)
    json_words = set(w)

  oov_words = json_words - (vocab | stem_vocab )
  print(len(oov_words))
  oov_dict_without_stem[file.name] = oov_words

1086
5504
1072
2270


In [167]:
folder =  Path('/content/drive/MyDrive/Capital_trials/test/postocr/without_stem')
oov_dict_without_stem = dict()

for file in folder.iterdir():
  with open(file, "r", encoding="utf-8") as f:
    w = json.load(f)
    json_words = set(w)

  oov_words = json_words - (vocab | stem_vocab | stem_vocab2)
  print(len(oov_words))
  oov_dict_without_stem[file.name] = oov_words

1078
5459
1064
2246


In [165]:
folder =  Path('/content/drive/MyDrive/Capital_trials/test/postocr/stem/')
oov_dict = dict()


for file in folder.iterdir():
  with open(file, "r", encoding="utf-8") as f:
    w = json.load(f)
    total_len = len(w)
    json_words = set(w)

  print('file name')
  print(file.name)

  oov_words = json_words - (stem_vocab | vocab | stem_vocab2)
  print('total length')
  print(total_len)

  print('num unique words')
  print(len(json_words))

  print('num unique out of vocab words')
  print(len(oov_words))
  oov_dict[file.name] = oov_words

file name
Christa Pike.json
total length
60646
num unique words
2697
num unique out of vocab words
289
file name
Fabian Hernandez Combined Transcript.json
total length
141899
num unique words
4301
num unique out of vocab words
558
file name
Christopher Sepulvado Combined Transcripto.json
total length
56251
num unique words
2816
num unique out of vocab words
222
file name
Socorro Caro.json
total length
803468
num unique words
8105
num unique out of vocab words
1634


In [173]:
oov_dict['Christopher Sepulvado Combined Transcript.json'] = oov_dict['Christopher Sepulvado Combined Transcripto.json']
del oov_dict['Christopher Sepulvado Combined Transcripto.json']

In [174]:
for file in folder.iterdir():
  with open(file, "r", encoding="utf-8") as f:
    words = json.load(f)

    set1 = oov_dict[file.name]
    real_words = [w for w in words if w not in set1]
    print(file.name)
    print('total words')
    print(len(words))
    print('total possible error words')
    print(len(words) - len(real_words))
    print('error rate')
    print( (len(words) - len(real_words)) / len(words))

Christa Pike.json
total words
73791
total possible error words
1496
error rate
0.02027347508503747
Socorro Caro.json
total words
957108
total possible error words
16235
error rate
0.016962558039427107
Christopher Sepulvado Combined Transcript.json
total words
64870
total possible error words
894
error rate
0.013781408971789734
Fabian Hernandez Combined Transcript.json
total words
171098
total possible error words
4578
error rate
0.026756595635249975


In [135]:
#remove apostraphe
for file in folder.iterdir():
  with open(file, "r", encoding="utf-8") as f:
    words = json.load(f)

    set1 = oov_dict[file.name]
    real_words = [w for w in words if w not in set1]
    print(file.name)
    print('total words')
    print(len(words))
    print('total possible error words')
    print(len(words) - len(real_words))
    print('error rate')
    print( (len(words) - len(real_words)) / len(words))

Christa Pike.json
total words
60646
total possible error words
2063
error rate
0.03401708274247271
Fabian Hernandez Combined Transcript.json
total words
141899
total possible error words
5275
error rate
0.037174328219367295
Christopher Sepulvado Combined Transcripto.json
total words
56251
total possible error words
1323
error rate
0.023519581874100016
Socorro Caro.json
total words
803468
total possible error words
20274
error rate
0.025233114448864172


In [121]:
oov_dict

{'Christa Pike.json': {'acumul',
  'ag',
  'ahold',
  'ain',
  'alcoa',
  'allright',
  'antipaladadin',
  'antipaladin',
  'anymor',
  'aq',
  'asto',
  'atchley',
  'athat',
  'athi',
  'attattorney',
  'avair',
  'barbi',
  'bbi',
  'beeler',
  'beingfrom',
  'belmont',
  'benchconfer',
  'benchconferencemr',
  'bernet',
  'beyonda',
  'bieber',
  'bik',
  'blew',
  'bohan',
  'bohanan',
  'boonan',
  'boyfriend',
  'brench',
  'brownan',
  'butcherknif',
  'cabbitre',
  'cabbr',
  'cabbre',
  'cabertre',
  'cameron',
  'carbertre',
  'carbtre',
  'carlson',
  'carrera',
  'carrthre',
  'carthre',
  'cassett',
  'caze',
  'cbabtre',
  'children',
  'christa',
  'clower',
  'colen',
  'colleleen',
  'collem',
  'couldn',
  'cqurt',
  'cra',
  'crabbet',
  'crabbre',
  'crabertre',
  'crabitre',
  'crabre',
  'crabt',
  'crabthre',
  'crabtr',
  'crabtre',
  'cravetre',
  'crbertre',
  'crbt',
  'crbtre',
  'crbtree',
  'crenshaw',
  'crorder',
  'croutre',
  'cumberland',
  'currre',

In [36]:
#sanity check file quality on a single example


In [ ]:
full_basic_clean(women_test1_post, output_folder_json, output_folder_stem):


In [ ]:
#stem the dictionary words


In [ ]:
#sanity check on a single example
# example_path = '/content/drive/MyDrive/Capital_trials/women/example/Angelina Rodriguez.txt'
# example_output = '/content/drive/MyDrive/Capital_trials/women/example/Angelina Rodriguez_bc.json'
# lower_tokenize_regex(example_path, example_output)

In [ ]:
# full_basic_clean('/content/drive/MyDrive/Capital_trials/women', '/content/drive/MyDrive/Capital_trials/basic_clean/women')
# full_basic_clean('/content/drive/MyDrive/Capital_trials/men', '/content/drive/MyDrive/Capital_trials/basic_clean/men')
# full_basic_clean('/content/drive/MyDrive/Capital_trials/joint', '/content/drive/MyDrive/Capital_trials/basic_clean/joint')

Remove outliers that are caused by missing space

In [ ]:
def likely_missing_space(token: str, vocab: nltk_vocab, min_part_len: int = 2) -> bool:
    t = re.sub(r"[^a-z']", "", token.lower())
    if len(t) < 2 * min_part_len or t in vocab:
        return False
    if t in vocab:  # already a known word
        return False

    for i in range(min_part_len, len(t) - min_part_len + 1):
        left, right = t[:i], t[i:]
        if left in vocab and right in vocab:
            return True
    return False



In [ ]:
def likely_missing_space_strict(token, vocab, word_freq, min_part_len=3, min_freq=1e-6, min_ratio=50):
    t = re.sub(r"[^a-z']", "", token.lower())
    if len(t) < 2 * min_part_len:
        return False, None

    whole_freq = word_freq.get(t, 0.0)
    if t in vocab and whole_freq >= min_freq:
        return False, None

    best = None
    best_score = 0.0
    for i in range(min_part_len, len(t) - min_part_len + 1):
        l, r = t[:i], t[i:]
        lf, rf = word_freq.get(l, 0.0), word_freq.get(r, 0.0)
        if l in vocab and r in vocab and lf >= min_freq and rf >= min_freq:
            score = lf * rf
            if score > best_score:
                best_score, best = score, (l, r)

    if best is None:
        return False, None

    if whole_freq > 0 and (best_score / whole_freq) < min_ratio:
        return False, None

    return True, best

In [ ]:
def seperate_missing_space_outliers(outliers, likely_missing_space_set, not_missing_space_set, vocab: nltk_vocab):
  for w in outliers:
      if likely_missing_space(w, vocab):
          likely_missing_space_set.add(w)
      else:
          not_missing_space_set.add(w)

  print("likely missing space:", len(likely_missing_space_set))
  print("not missing space:", len(not_missing_space_set))

In [ ]:
likely_missing_space_w = set()
remaining_outliers_w = set()

likely_missing_space_m = set()
remaining_outliers_m = set()


In [ ]:
seperate_missing_space_outliers(outlier_w, likely_missing_space_w, remaining_outliers_w, nltk_vocab )


likely missing space: 25520
not missing space: 42101


In [ ]:
seperate_missing_space_outliers(outlier_m, likely_missing_space_m, remaining_outliers_m, nltk_vocab )


likely missing space: 74971
not missing space: 224559


In [ ]:
remaining_outliers_w_m = remaining_outliers_w| remaining_outliers_m
len(remaining_outliers_w_m)

251741

In [ ]:

print(f"Number womens unqiue words {len(w_dict)} Number of womens outliers {len(remaining_outliers_w)}")

print(f"Number mens unqiue words {len(m_dict)} Number of mens outliers{len(remaining_outliers_m)}")

print(f"Total unqiue words {len(w_dict | m_dict )} total outliers {len(remaining_outliers_w_m)}")

Number womens unqiue words 91764 Number of womens outliers 42101
Number mens unqiue words 324949 Number of mens outliers224559
Total unqiue words 370303 total outliers 251741


In [ ]:
likely_missing_space_w_m = likely_missing_space_m | likely_missing_space_w
len(likely_missing_space_w_m)

88628

For ourliers caused by missing space, create map of splits

In [ ]:
def best_two_word_split(token: str, vocab: set[str], word_freq: dict[str, float] | None = None, min_part_len: int = 2):
    t = re.sub(r"[^a-z']", "", token.lower())
    n = len(t)
    if n < 2 * min_part_len or t in vocab:
        return None

    best = None
    best_score = float("-inf")

    for i in range(min_part_len, n - min_part_len + 1):
        left, right = t[:i], t[i:]
        if left in vocab and right in vocab:
            if word_freq:
                score = word_freq.get(left, 1e-12) * word_freq.get(right, 1e-12)
            else:
                score = -abs(len(left) - len(right))  # fallback heuristic
            if score > best_score:
                best_score = score
                best = (left, right)

    return best


In [ ]:
sample_missing_space = random.sample(list(likely_missing_space_w_m), 2000)

In [ ]:
start = time.time()

split_map = {}
for w in likely_missing_space_w_m:
    split = best_two_word_split(w, nltk_vocab, word_freq=None)
    if split is not None:
        split_map[w] = split
split_text_map = {k: f"{v[0]} {v[1]}" for k, v in split_map.items()}

print(f"splits for {len(split_map)} tokens")

end = time.time()
print(f"took  {end- start} sec")

splits for 88628 tokens
took  1.5462627410888672 sec


In [ ]:
len(split_map.keys())

88628

In [ ]:
with open('/content/drive/MyDrive/Capital_trials/basic_clean/spell/replace_missing_space.json', "w", encoding="utf-8") as outfile:
  json.dump(split_map, outfile)

Spellcheck outliers not split by space

In [ ]:
def spellcheck(remaining_outliers):
  spellchecked_likely_outlier = set()
  spellchecked_likely_word = set()

  for word in remaining_outliers:
    freq = zipf_frequency(word, 'en')
    if freq < 2.5:
      spellchecked_likely_outlier.add(word)
    else:
      spellchecked_likely_word.add(word)


  return spellchecked_likely_outlier, spellchecked_likely_word

In [ ]:
spellchecked_likely_outlier_w , spellchecked_likely_word_w = spellcheck(remaining_outliers_w)

spellchecked_likely_outlier_m , spellchecked_likely_word_m = spellcheck(remaining_outliers_m)


In [ ]:
print(f"Number womens unqiue words {len(w_dict)} Number of womens outliers remaining {len(spellchecked_likely_outlier_w)}")

print(f"Number mens unqiue words {len(m_dict)} Number of mens outliers remaining {len(spellchecked_likely_outlier_m)}")

print(f"Total unqiue words {len(w_dict | m_dict )} total outliers remaining {len(spellchecked_likely_outlier_m | spellchecked_likely_outlier_w)}")

Number womens unqiue words 91764 Number of womens outliers remaining 34162
Number mens unqiue words 324949 Number of mens outliers remaining 215966
Total unqiue words 370303 total outliers remaining 241940


In [ ]:
#import past checkpoints of previous runs of create_replacement_dict to avoid processing seen words
with open('/content/drive/MyDrive/Capital_trials/basic_clean/spell/replace_most_likely.json', 'r') as file:
  dict1 = json.load(file)

with open('/content/drive/MyDrive/Capital_trials/basic_clean/spell/replace_missing_space.json', 'r') as file:
  dict2 = json.load(file)

with open('/content/drive/MyDrive/Capital_trials/basic_clean/spell/replace_sample.json', 'r') as file:
  dict3 = json.load(file)

In [ ]:

len(dict1.keys())


1750

In [ ]:
len(dict2.keys())

88628

In [ ]:
len(dict3.keys())

261

In [ ]:
replacement_dict_partial = dict1.keys() | dict2.keys() |dict3.keys()
len(replacement_dict_partial)

90554

In [ ]:
remaining_to_replace = spellchecked_likely_outlier_m -replacement_dict_partial

In [ ]:
len(remaining_to_replace)

215928

In [ ]:
save_remaining_to_replacesave_remaining_to_replace = {"outliers_to_fix": list(remaining_to_replace)}
with open('/content/drive/MyDrive/Capital_trials/basic_clean/spell/outliers_to_replace.json', "w", encoding="utf-8") as outfile:
  json.dump(save_remaining_to_replace, outfile)

Check for multiple words together and create replacement dictionary

In [ ]:
import re
from functools import lru_cache

def word_break_best(
    token: str,
    vocab: set[str],
    word_freq: dict[str, float] | None = None,
    min_len: int = 2,
    max_parts: int = 6,
) -> list[str] | None:
    """
    Split a token into the most likely sequence of words.
    Returns list of words, or None if no valid split.
    """
    t = re.sub(r"[^a-z']", "", token.lower())
    n = len(t)
    if n < min_len:
        return None

    # If already a known word, keep as-is
    if t in vocab:
        return [t]

    # Default frequency: treat all words equally
    def score_word(w: str) -> float:
        if word_freq is None:
            return 1.0
        return max(word_freq.get(w, 1e-12), 1e-12)

    @lru_cache(None)
    def dp(i: int, parts_used: int):
        if i == n:
            return (0.0, [])  # log-score, words
        if parts_used >= max_parts:
            return None

        best = None
        # try next word from i:j
        for j in range(i + min_len, n + 1):
            w = t[i:j]
            if w not in vocab:
                continue
            rest = dp(j, parts_used + 1)
            if rest is None:
                continue
            rest_score, rest_words = rest
            # use log-like additive scoring via multiplication proxy
            s = rest_score + (score_word(w))
            cand = (s, [w] + rest_words)
            if best is None or cand[0] > best[0]:
                best = cand
        return best

    result = dp(0, 0)
    return result[1] if result else None


Create Dictionary of misspelled words and possible replacements, save words that do not have good replacement options seperately


In [ ]:
def create_replacement_dict(outlier_set, vocab):
  replacement_dict = {}
  no_replacement_available = set()

  for word in (outlier_set):

    most_likely = spell.correction(word)

    if most_likely is not None:

      replacement_dict[word] = most_likely
    else:
      result = word_break_best(word, vocab)

      if result is not None:
        replacement_dict[word] = result
      else:
        no_replacement_available.add(word)

  return replacement_dict,no_replacement_available

In [ ]:
#create a smaller subset of outliers to test
sample_n = 500

sample = random.sample(list(spellchecked_likely_outlier_m | spellchecked_likely_outlier_w), min(sample_n, len(list(spellchecked_likely_outlier_m | spellchecked_likely_outlier_w))))


In [ ]:
with open('/content/drive/MyDrive/Capital_trials/basic_clean/spell/replace_sample.json', "w", encoding="utf-8") as outfile:
  json.dump(replacement_dict, outfile)

In [ ]:
import time

def create_replacement_dict_timed(outlier_set, vocab, max_items=None):
    replacement_dict = {}
    no_replacement_available = set()

    n = 0
    used_fallback = 0
    t0 = time.time()

    for word in outlier_set:
        n += 1
        if max_items and n > max_items:
            break

        most_likely = spell.correction(word)

        if most_likely is not None:
            replacement_dict[word] = most_likely
        else:
            used_fallback += 1
            result = word_break_best(word, vocab)
            if result is not None:
                replacement_dict[word] = result
            else:
                no_replacement_available.add(word)

        if n % 200 == 0:
            dt = time.time() - t0
            print(f"{n} done | {dt:.1f}s | fallback={used_fallback} ({used_fallback/n:.1%})")

    return replacement_dict, no_replacement_available
